# Energy Analytics - Exploratory Data Analysis

This notebook provides an interactive analysis of the energy operations data,
including production metrics, equipment health, and emissions monitoring.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from datetime import datetime

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Database connection
conn_params = {
    'host': os.environ.get('POSTGRES_HOST', 'localhost'),
    'port': int(os.environ.get('POSTGRES_PORT', '5432')),
    'database': os.environ.get('POSTGRES_DB', 'energy_analytics'),
    'user': os.environ.get('POSTGRES_USER', 'postgres'),
    'password': os.environ.get('POSTGRES_PASSWORD', 'postgres'),
}

def query_db(sql):
    """Execute SQL query and return DataFrame."""
    with psycopg2.connect(**conn_params) as conn:
        return pd.read_sql(sql, conn)

print(f"Analysis Date: {datetime.now():%Y-%m-%d %H:%M}")

## 1. Production Overview

In [ ]:
# Load production data
production_df = query_db("""
    SELECT 
        well_id,
        well_name,
        operator,
        basin,
        production_date,
        oil_bbl,
        gas_mcf,
        water_bbl,
        boe_total as boe,
        water_cut_pct,
        gor_mcf_bbl as gas_oil_ratio
    FROM staging_analytics.fct_well_performance_daily
    ORDER BY production_date
""")

print(f"Total production records: {len(production_df):,}")
print(f"Date range: {production_df['production_date'].min()} to {production_df['production_date'].max()}")
print(f"Unique wells: {production_df['well_id'].nunique()}")
production_df.head()

In [ ]:
# Production summary statistics
print("\nProduction Statistics:")
production_df[['oil_bbl', 'gas_mcf', 'water_bbl', 'boe']].describe()

In [ ]:
# Production by operator
operator_production = production_df.groupby('operator').agg({
    'oil_bbl': 'sum',
    'gas_mcf': 'sum',
    'boe': 'sum',
    'well_id': 'nunique'
}).rename(columns={'well_id': 'well_count'}).sort_values('boe', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
operator_production['boe'].head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Total BOE')
ax.set_ylabel('Operator')
ax.set_title('Top 10 Operators by Total BOE Production')
plt.tight_layout()
plt.show()

In [ ]:
# Production trends over time
if 'production_date' in production_df.columns:
    # Convert to datetime and aggregate
    production_df['production_date'] = pd.to_datetime(production_df['production_date'])
    daily_production = production_df.groupby('production_date').agg({
        'oil_bbl': 'sum',
        'gas_mcf': 'sum',
        'boe': 'sum'
    }).sort_index()
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    axes[0].plot(daily_production.index, daily_production['oil_bbl'], color='green', marker='o', markersize=3)
    axes[0].set_ylabel('Oil (BBL)')
    axes[0].set_title('Daily Production Trends')
    
    axes[1].plot(daily_production.index, daily_production['gas_mcf'], color='orange', marker='o', markersize=3)
    axes[1].set_ylabel('Gas (MCF)')
    
    axes[2].plot(daily_production.index, daily_production['boe'], color='steelblue', marker='o', markersize=3)
    axes[2].set_ylabel('BOE')
    axes[2].set_xlabel('Date')
    
    # Format x-axis
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 2. Equipment Health Analysis

In [ ]:
# Load equipment data
equipment_df = query_db("""
    SELECT 
        equipment_id,
        well_id,
        equipment_type,
        manufacturer,
        health_score,
        health_category,
        equipment_status,
        maintenance_priority_score,
        days_until_maintenance,
        is_maintenance_overdue,
        runtime_hours
    FROM staging_analytics.fct_equipment_health
""")

print(f"Total equipment records: {len(equipment_df):,}")
equipment_df.head()

In [ ]:
# Equipment health distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Health category distribution
health_counts = equipment_df['health_category'].value_counts()
colors = ['#2ecc71', '#27ae60', '#f39c12', '#e74c3c', '#c0392b']
health_order = ['Excellent', 'Good', 'Fair', 'Poor', 'Critical']
health_counts = health_counts.reindex([c for c in health_order if c in health_counts.index])
health_counts.plot(kind='pie', ax=axes[0], colors=colors[:len(health_counts)], autopct='%1.1f%%')
axes[0].set_title('Equipment Health Distribution')
axes[0].set_ylabel('')

# Health score by equipment type
equipment_df.boxplot(column='health_score', by='equipment_type', ax=axes[1])
axes[1].set_title('Health Score by Equipment Type')
axes[1].set_xlabel('Equipment Type')
axes[1].set_ylabel('Health Score')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Equipment requiring attention
critical_equipment = equipment_df[
    (equipment_df['health_category'].isin(['Poor', 'Critical'])) |
    (equipment_df['is_maintenance_overdue'] == True)
].sort_values('maintenance_priority_score', ascending=False)

print(f"Equipment requiring attention: {len(critical_equipment)}")
critical_equipment[['equipment_id', 'well_id', 'equipment_type', 'health_score', 
                    'health_category', 'is_maintenance_overdue', 'maintenance_priority_score']].head(10)

## 3. Emissions Analysis

In [ ]:
# Load emissions data
emissions_df = query_db("""
    SELECT 
        well_id,
        measurement_date,
        methane_kg,
        co2_kg,
        voc_kg,
        co2_equivalent_kg,
        flare_volume_mcf,
        leak_detected,
        environmental_risk as risk_level
    FROM staging_analytics.fct_emissions_daily
    ORDER BY measurement_date
""")

print(f"Total emissions records: {len(emissions_df):,}")
emissions_df.head()

In [ ]:
# Emissions summary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk level distribution
risk_counts = emissions_df['risk_level'].value_counts()
risk_colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
risk_counts.plot(kind='bar', ax=axes[0], color=[risk_colors.get(r, 'gray') for r in risk_counts.index])
axes[0].set_title('Emissions Risk Level Distribution')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Leak detection
leak_counts = emissions_df['leak_detected'].value_counts()
leak_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Leak Detection Status')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# CO2 equivalent emissions by well (top emitters)
well_emissions = emissions_df.groupby('well_id').agg({
    'co2_equivalent_kg': 'sum',
    'methane_kg': 'sum',
    'flare_volume_mcf': 'sum'
}).sort_values('co2_equivalent_kg', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
well_emissions['co2_equivalent_kg'].head(15).plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Total CO2 Equivalent (kg)')
ax.set_ylabel('Well ID')
ax.set_title('Top 15 Wells by CO2 Equivalent Emissions')
plt.tight_layout()
plt.show()

## 4. Operator Summary

In [ ]:
# Load operator summary
operator_df = query_db("""
    SELECT *
    FROM staging_analytics.dim_operator_summary
    ORDER BY total_boe_30d DESC
""")

print(f"Total operators: {len(operator_df)}")
operator_df

In [ ]:
# Operator comparison
if len(operator_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Wells count
    operator_df.set_index('operator')['total_wells'].plot(
        kind='bar', ax=axes[0, 0], color='steelblue'
    )
    axes[0, 0].set_title('Total Wells by Operator')
    axes[0, 0].set_ylabel('Well Count')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # BOE Production
    operator_df.set_index('operator')['total_boe_30d'].plot(
        kind='bar', ax=axes[0, 1], color='green'
    )
    axes[0, 1].set_title('Total BOE by Operator (30 days)')
    axes[0, 1].set_ylabel('BOE')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Revenue
    operator_df.set_index('operator')['total_oil_bbl_30d'].plot(
        kind='bar', ax=axes[1, 0], color='darkgreen'
    )
    axes[1, 0].set_title('Oil Production by Operator (30 days)')
    axes[1, 0].set_ylabel('Oil (BBL)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Equipment Health
    operator_df.set_index('operator')['avg_equipment_health'].plot(
        kind='bar', ax=axes[1, 1], color='purple'
    )
    axes[1, 1].set_title('Average Equipment Health by Operator')
    axes[1, 1].set_ylabel('Health Score')
    axes[1, 1].axhline(y=70, color='red', linestyle='--', label='Threshold')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 5. Key Findings Summary

In [ ]:
# Generate key findings
print("=" * 60)
print("KEY FINDINGS SUMMARY")
print("=" * 60)

# Production highlights
print("\n📊 PRODUCTION METRICS:")
print(f"  • Total wells analyzed: {production_df['well_id'].nunique():,}")
print(f"  • Total oil production: {production_df['oil_bbl'].sum():,.0f} BBL")
print(f"  • Total gas production: {production_df['gas_mcf'].sum():,.0f} MCF")
print(f"  • Total BOE: {production_df['boe'].sum():,.0f}")

# Equipment highlights
print("\n🔧 EQUIPMENT HEALTH:")
critical_count = len(equipment_df[equipment_df['health_category'].isin(['Poor', 'Critical'])])
print(f"  • Total equipment: {len(equipment_df):,}")
print(f"  • Average health score: {equipment_df['health_score'].mean():.1f}")
print(f"  • Critical/Poor equipment: {critical_count} ({100*critical_count/len(equipment_df):.1f}%)")

# Emissions highlights
print("\n🌿 EMISSIONS:")
leak_count = len(emissions_df[emissions_df['leak_detected'] == True])
print(f"  • Total CO2 equivalent: {emissions_df['co2_equivalent_kg'].sum():,.0f} kg")
print(f"  • Detected leaks: {leak_count}")
high_risk = len(emissions_df[emissions_df['risk_level'] == 'High'])
print(f"  • High risk readings: {high_risk}")

print("\n" + "=" * 60)